In [2]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from qopt_funcs import *
from network_funcs import *

In [3]:
# MAKE ALL THE SPACING CONSISTENT
PARAMS = {
    'p_det': 0.95,          # Detector efficiency
    'alpha': 0.18,          # Fiber loss (dB/km)
    'q_0': 0.01,            # Baseline QBER (Check)
    'nu': 10**9,            # Repetition rate (1 GHz) !! Check
    'R_dark': 100,          # Dark count rate (Hz)
    'delta_det': 100.E-12,  # Time gate duration (s) !! Check
    'p_pair': 0.05,         # Pair generation probability
    'eta_c': 0.8,           # Source to fiber coupling

}

In [51]:
N=25
beta = 2.6261                 # \beta param of S2 model
mu = 0.0233
A, dist = S2_graph_definite_N(N, beta, mu, D=2, sample_from_file=False, return_coords=False)

In [52]:
def build_prob_matrix(A, dist, architecture='node'):
    # Check
    if architecture == 'node':
        dist_A=np.zeros(np.shape(dist))
        dist_B=dist
    elif architecture == 'midpoint':
        dist_A=dist/2
        dist_B=dist/2
    else:
        raise ValueError('architecture must be node or midpoint')
    eta_A_mtx = PARAMS['eta_c'] * PARAMS['p_det'] * 10**(-PARAMS['alpha'] * dist_A / 10)*A # Multiply by Adjacency matrix to keep only existing edges
    eta_B_mtx = PARAMS['eta_c'] * PARAMS['p_det'] * 10**(-PARAMS['alpha'] * dist_B / 10)*A
    P_ent_matrix = PARAMS['p_pair'] * eta_A_mtx * eta_B_mtx
    return P_ent_matrix, eta_A_mtx, eta_B_mtx

Probs_mtx, eta_A_mtx, eta_B_mtx = build_prob_matrix(A, dist, architecture='midpoint')

In [58]:
def optimal_quantum_repeater_path(Probs_mtx, source, target=None,  P_BSM=1): # structured like nx.single_source_dijkstra
    W = np.zeros(np.shape(Probs_mtx))
    none_zero_edges = Probs_mtx>0
    W[none_zero_edges] = -np.log2(Probs_mtx[none_zero_edges]) - np.log2(P_BSM)
    G = nx.from_numpy_array(W)
    weights, paths = nx.single_source_dijkstra(G, source, target, weight='weight')
    #weights = np.exp(weights + np.log2(P_BSM))   # to avoid overcounting the Bell state success probability
    return weights, paths

weights, path = optimal_quantum_repeater_path(Probs_mtx, 0, target=24)
path

[0, 13, 16, 21, 24]

In [59]:
def calculate_expected_sequential_time(path, Probs_mtx, P_BSM=1):
    # Could include calculation of intermediate Q (think about how this is done)_link
    a = path[0] # First Node
    b = path[1] # Second Node
    T=1/Probs_mtx[a, b]
    for i in range(2,len(path)):
        a = path[i-1]
        b = path[i]
        T_i=1/Probs_mtx[a, b]
        T=(T+T_i)/P_BSM
    return T

total_time = calculate_expected_sequential_time(path, Probs_mtx)
print(total_time)

143.3022862717663


## Post processing functions

In [60]:
def entanglement_generation_rate(total_time):
    return PARAMS['nu']/total_time

def QBER(eta_A_mtx, eta_B_mtx, path):
    p_darkcount = PARAMS['R_dark']*PARAMS['delta_det']
    W=1
    for i in range(1, len(path)):
        a = path[i-1]
        b = path[i]
        eta_A = eta_A_mtx[a,b]
        eta_B = eta_B_mtx[a,b]
        p_acc = PARAMS['p_pair']*eta_A*(1-eta_B)*p_darkcount+PARAMS['p_pair']*eta_B*(1-eta_A)*p_darkcount+p_darkcount**2
        p_det = PARAMS['p_pair']*eta_A*eta_B
        Q_i = (p_det*(PARAMS['q_0']+PARAMS['p_pair']/2)+0.5*p_acc)/(p_det + p_acc)
        W_i = 1-2*Q_i
        W = W*W_i
    return (1-W)/2
R_ent = entanglement_generation_rate(total_time)
Q = QBER(eta_A_mtx, eta_B_mtx, path)
print(Q)

0.12597400511984985


In [61]:
def secret_key_rate_BBM92(R_ent, Q):
    H = binary_entropy(Q)
    return R_ent*(1-2*H)
secret_key_rate_BBM92(R_ent*0.5, Q)

np.float64(-323041.4503016836)

Look at distances and see why I am getting negative values so easily for more than 3 steps!!!!